# Domain-adapted NLLB RNMT v3 — development-only LoRA experiment

**Protocol:** `nllb-v3-lora-dev-v1`  
**Frozen protocol SHA-256:** `242297bc42129587d84a9a3ca51a0b6105ac42181ffb706cc021202e3703a9d9`  
**Prepared:** 4 August 2026

This notebook runs the pre-specified three-seed, group-balanced LoRA
adaptation of `facebook/nllb-200-distilled-600M` for maternal-health
Twi → English translation. It evaluates both validated gold Twi and
frozen raw MMS transcriptions. It does **not** read or evaluate the
sealed test partition, call OpenAI, change the SBLLM, or deploy a model.

Run the cells in order. The notebook deliberately aborts for an input
hash mismatch, train/development leakage, tokenizer truncation, or a
GPU other than A100/H100. Outputs and checkpoints are written directly
to Google Drive so a Colab disconnect does not erase the research record.

## 1. Install the frozen software layer

PyTorch and CUDA are supplied by the selected Colab image and are not
silently replaced. Their exact versions are recorded later. The
experiment-specific libraries below are pinned by the frozen protocol.

In [3]:
# Install the exact user-space packages declared in the frozen protocol.
%pip install -q --upgrade \
  transformers==4.57.1 \
  peft==0.17.1 \
  accelerate==1.11.0 \
  datasets==4.3.0 \
  evaluate==0.4.6 \
  sacrebleu==2.5.1 \
  sentencepiece==0.2.1 \
  scikit-learn==1.7.2

## 2. Mount Drive and create an immutable run folder

The expected input directory is:

`/content/drive/MyDrive/Akan_ASR_PhD_Experiments/03_Adaptation/nllb_v3_2026-08-04/inputs`

If your Drive folder has a different name, change only `PROJECT_ROOT`
before running this cell and record the change in the run notes.

In [4]:
# Mount the researcher's authenticated Google Drive.
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

# Import standard-library utilities before configuring the run.
import atexit
import gc
import hashlib
import importlib.metadata
import json
import os
import platform
import random
import re
import shutil
import subprocess
import sys
import time
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

# Import the pinned scientific stack.
import numpy as np
import pandas as pd
import torch

# Freeze all project and run paths in one visible location.
PROJECT_ROOT = Path("/content/drive/MyDrive/Akan_ASR_PhD_Experiments")
INPUT_DIR = PROJECT_ROOT / "03_Adaptation" / "nllb_v3_2026-08-04" / "inputs"
RUN_PARENT = PROJECT_ROOT / "03_Adaptation" / "nllb_v3_2026-08-04"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ_nllb_v3_lora_dev_v1")
RUN_DIR = RUN_PARENT / "runs" / RUN_ID
OUTPUT_DIR = RUN_DIR / "outputs"
ADAPTER_DIR = RUN_DIR / "adapters"
CHECKPOINT_DIR = RUN_DIR / "checkpoints"
LOG_DIR = RUN_DIR / "logs"
REVIEW_DIR = RUN_DIR / "human_review"
ENV_DIR = RUN_DIR / "environment"

# Create Drive-backed folders before any expensive computation starts.
for directory in [OUTPUT_DIR, ADAPTER_DIR, CHECKPOINT_DIR, LOG_DIR, REVIEW_DIR, ENV_DIR]:
    directory.mkdir(parents=True, exist_ok=True)

# Keep the public base-model cache on fast ephemeral storage; immutable
# model revision and output hashes preserve provenance.
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

# A small status record lets a disconnected runtime leave an honest trail.
RUN_STATE = {"complete": False, "run_id": RUN_ID, "started_utc": datetime.now(timezone.utc).isoformat()}

def write_json(path, payload):
    '''Write deterministic, human-readable JSON to Drive.'''
    path.write_text(json.dumps(payload, ensure_ascii=False, indent=2, sort_keys=True), encoding="utf-8")

def write_incomplete_record():
    '''Record that the notebook ended before the final completion cell.'''
    if not RUN_STATE["complete"]:
        write_json(
            RUN_DIR / "RUN_INCOMPLETE.json",
            {
                **RUN_STATE,
                "status": "INCOMPLETE",
                "meaning": "The final completion cell did not run; do not use this run for a claim or deployment.",
                "ended_utc": datetime.now(timezone.utc).isoformat(),
            },
        )

atexit.register(write_incomplete_record)
print(f"Run folder: {RUN_DIR}")

Mounted at /content/drive
Run folder: /content/drive/MyDrive/Akan_ASR_PhD_Experiments/03_Adaptation/nllb_v3_2026-08-04/runs/20260804T191924Z_nllb_v3_lora_dev_v1


## 3. Verify hardware, package versions, environment, and immutable inputs

This cell is intentionally strict. A failed assertion protects the
experiment from quietly running on the wrong data or environment.

In [5]:
# Import model and evaluation libraries only after the pinned install.
import accelerate
import datasets
import evaluate
import peft
import sacrebleu
import sklearn
import transformers
from sacrebleu.metrics import BLEU, CHRF

# Declare the package versions fixed in the protocol.
EXPECTED_PACKAGES = {
    "transformers": "4.57.1",
    "peft": "0.17.1",
    "accelerate": "1.11.0",
    "datasets": "4.3.0",
    "evaluate": "0.4.6",
    "sacrebleu": "2.5.1",
    "sentencepiece": "0.2.1",
    "scikit-learn": "1.7.2",
}

# Read installed versions directly from package metadata.
installed_packages = {
    name: importlib.metadata.version(name)
    for name in EXPECTED_PACKAGES
}

# Refuse to continue if Colab resolved a different declared version.
assert installed_packages == EXPECTED_PACKAGES, {
    "expected": EXPECTED_PACKAGES,
    "installed": installed_packages,
}

# Require the accelerator class frozen before training.
assert torch.cuda.is_available(), "CUDA GPU is required."
gpu_name = torch.cuda.get_device_name(0)
assert ("A100" in gpu_name) or ("H100" in gpu_name), f"A100/H100 required; found {gpu_name}."
assert torch.cuda.is_bf16_supported(), f"bfloat16 is not supported by {gpu_name}."

# Configure reproducibility safeguards; CUDA generation may still have
# platform-level nondeterminism, which is why three seeds are retained.
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True
torch.backends.cuda.matmul.allow_tf32 = True
torch.set_float32_matmul_precision("high")
torch.use_deterministic_algorithms(True, warn_only=True)

# Declare immutable model, protocol, and file identifiers.
PROTOCOL_ID = "nllb-v3-lora-dev-v1"
PROTOCOL_SHA256 = "242297bc42129587d84a9a3ca51a0b6105ac42181ffb706cc021202e3703a9d9"
BASE_MODEL_ID = "facebook/nllb-200-distilled-600M"
BASE_MODEL_REVISION = "f8d333a098d19b4fd9a8b18f94170487ad3f821d"
MMS_MODEL_ID = "facebook/mms-1b-all"
MMS_MODEL_REVISION = "3d33597edbdaaba14a8e858e2c8caa76e3cec0cd"
SOURCE_LANG = "twi_Latn"
TARGET_LANG = "eng_Latn"
SOURCE_MAX_TOKENS = 192
TARGET_MAX_TOKENS = 192
SEEDS = [17, 29, 47]
BOOTSTRAP_SEED = 20260804
BOOTSTRAP_REPLICATES = 20_000

# Map every required local input to its frozen SHA-256 digest.
EXPECTED_HASHES = {
    "train_pairs_group_balanced_v1.csv": "6672887fe9117e9f14ec085ef79533b5c148b85d10043d449b811d4a06a1bc37",
    "dev_pairs_gold_and_mms_v1.csv": "cb202d42d8a0d079f68515a4effd536be2ee91cea5e4689e2f55251a5c67626a",
    "DATASET_MANIFEST_v1.json": "c86ee991c6d588e2dd568381a2e6a366964338b3fda2c23300713da6a2462c47",
    "NLLB_V3_FROZEN_CLINICAL_LEXICON_v1.json": "a0532f1a5124dd48e90f9f4697cc8a9ddb7ac99048244c2c70dcb6a7727b23aa",
}

def sha256_file(path, chunk_size=1024 * 1024):
    '''Calculate a file digest without loading the entire file into memory.'''
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        while True:
            chunk = handle.read(chunk_size)
            if not chunk:
                break
            digest.update(chunk)
    return digest.hexdigest()

# Check existence and content identity before reading the data.
observed_hashes = {}
for filename, expected_hash in EXPECTED_HASHES.items():
    path = INPUT_DIR / filename
    assert path.exists(), f"Required input is missing: {path}"
    observed_hashes[filename] = sha256_file(path)
    assert observed_hashes[filename] == expected_hash, {
        "file": filename,
        "expected": expected_hash,
        "observed": observed_hashes[filename],
    }

# Save a complete execution environment before model download or training.
environment = {
    "protocol_id": PROTOCOL_ID,
    "protocol_sha256": PROTOCOL_SHA256,
    "run_id": RUN_ID,
    "utc": datetime.now(timezone.utc).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "torch": torch.__version__,
    "cuda_runtime": torch.version.cuda,
    "cudnn": torch.backends.cudnn.version(),
    "gpu_name": gpu_name,
    "gpu_count": torch.cuda.device_count(),
    "installed_packages": installed_packages,
    "input_hashes": observed_hashes,
    "base_model_id": BASE_MODEL_ID,
    "base_model_revision": BASE_MODEL_REVISION,
    "mms_model_id": MMS_MODEL_ID,
    "mms_model_revision": MMS_MODEL_REVISION,
}
write_json(ENV_DIR / "ENVIRONMENT.json", environment)

# Preserve a full package freeze and GPU report as plain-text evidence.
(ENV_DIR / "pip_freeze.txt").write_text(
    subprocess.check_output([sys.executable, "-m", "pip", "freeze"], text=True),
    encoding="utf-8",
)
(ENV_DIR / "nvidia_smi.txt").write_text(
    subprocess.check_output(["nvidia-smi"], text=True),
    encoding="utf-8",
)
print(json.dumps(environment, indent=2))

{
  "protocol_id": "nllb-v3-lora-dev-v1",
  "protocol_sha256": "242297bc42129587d84a9a3ca51a0b6105ac42181ffb706cc021202e3703a9d9",
  "run_id": "20260804T191924Z_nllb_v3_lora_dev_v1",
  "utc": "2026-08-04T19:19:45.812955+00:00",
  "python": "3.12.13 (main, Mar  4 2026, 09:23:07) [GCC 11.4.0]",
  "platform": "Linux-6.6.122+-x86_64-with-glibc2.35",
  "torch": "2.11.0+cu128",
  "cuda_runtime": "12.8",
  "cudnn": 91900,
  "gpu_name": "NVIDIA A100-SXM4-40GB",
  "gpu_count": 1,
  "installed_packages": {
    "transformers": "4.57.1",
    "peft": "0.17.1",
    "accelerate": "1.11.0",
    "datasets": "4.3.0",
    "evaluate": "0.4.6",
    "sacrebleu": "2.5.1",
    "sentencepiece": "0.2.1",
    "scikit-learn": "1.7.2"
  },
  "input_hashes": {
    "train_pairs_group_balanced_v1.csv": "6672887fe9117e9f14ec085ef79533b5c148b85d10043d449b811d4a06a1bc37",
    "dev_pairs_gold_and_mms_v1.csv": "cb202d42d8a0d079f68515a4effd536be2ee91cea5e4689e2f55251a5c67626a",
    "DATASET_MANIFEST_v1.json": "c86ee991c6d5

## 4. Load and audit the frozen train/development boundary

The test partition is absent by design. The development file contains
paired gold and raw MMS source columns for the same approved records.

In [6]:
# Load the frozen materialised datasets and their manifest.
train_df = pd.read_csv(INPUT_DIR / "train_pairs_group_balanced_v1.csv")
dev_df = pd.read_csv(INPUT_DIR / "dev_pairs_gold_and_mms_v1.csv")
manifest = json.loads((INPUT_DIR / "DATASET_MANIFEST_v1.json").read_text(encoding="utf-8"))
clinical_lexicon = json.loads(
    (INPUT_DIR / "NLLB_V3_FROZEN_CLINICAL_LEXICON_v1.json").read_text(encoding="utf-8")
)

# Verify row and group counts against the frozen manifest.
assert len(train_df) == 7240 == manifest["train_rows"]
assert train_df["content_group_id"].nunique() == 2139 == manifest["train_groups"]
assert len(dev_df) == 1558 == manifest["dev_rows"]
assert dev_df["content_group_id"].nunique() == 458 == manifest["dev_groups"]

def strict_boolean_series(series):
    '''Convert only explicit True/False values; reject every other token.'''
    normalized = series.astype(str).str.strip().str.casefold()
    allowed = {"true", "false"}
    assert set(normalized.unique()).issubset(allowed), sorted(normalized.unique())
    return normalized.map({"true": True, "false": False})

# Recompute the central leakage protections instead of trusting a label.
train_groups = set(train_df["content_group_id"].astype(str))
dev_groups = set(dev_df["content_group_id"].astype(str))
assert train_groups.isdisjoint(dev_groups), "Train/development semantic-group leakage detected."
assert not strict_boolean_series(dev_df["exact_gold_source_seen_in_train"]).any()
assert not strict_boolean_series(dev_df["exact_mms_source_seen_in_train"]).any()
assert not strict_boolean_series(dev_df["gold_source_conflicting_train_target"]).any()
assert not strict_boolean_series(dev_df["mms_source_conflicting_train_target"]).any()
assert manifest["test_rows_exported"] == 0
assert manifest["test_predictions_read"] == 0

# Verify each sampling weight implements 1 / number of variants in group.
expected_weights = 1.0 / train_df["group_variant_count"].astype(float)
assert np.allclose(train_df["sampling_weight"].astype(float), expected_weights, atol=1e-12)

# Convert the paired development columns into an explicit long form.
dev_gold = dev_df.copy()
dev_gold["input_condition"] = "gold"
dev_gold["source_twi"] = dev_gold["source_gold_twi"].fillna("").astype(str)
dev_mms = dev_df.copy()
dev_mms["input_condition"] = "mms"
dev_mms["source_twi"] = dev_mms["source_mms_twi"].fillna("").astype(str)
dev_long = pd.concat([dev_gold, dev_mms], ignore_index=True)
dev_long = dev_long.rename(columns={"target_english": "reference_english"})

# Preserve an immutable copy of the exact run inputs inside the run folder.
for filename in EXPECTED_HASHES:
    shutil.copy2(INPUT_DIR / filename, RUN_DIR / filename)

# Save a compact integrity report for later audit.
data_audit = {
    "train_rows": len(train_df),
    "train_groups": train_df["content_group_id"].nunique(),
    "dev_rows": len(dev_df),
    "dev_long_rows": len(dev_long),
    "dev_original_groups": dev_df["content_group_id"].nunique(),
    "train_dev_group_overlap": len(train_groups & dev_groups),
    "dev_conditions": dev_long["input_condition"].value_counts().to_dict(),
    "test_rows_loaded": 0,
}
write_json(OUTPUT_DIR / "DATA_INTEGRITY_AUDIT.json", data_audit)
display(pd.DataFrame([data_audit]))

,train_rows,train_groups,dev_rows,dev_long_rows,dev_original_groups,train_dev_group_overlap,dev_conditions,test_rows_loaded
0,7240,2139,1558,3116,458,0,"{'gold': 1558, 'mms': 1558}",0


## 5. Tokenizer audit — abort on any truncation

Token lengths are measured without truncation for every train and
development source/reference. Training cannot begin if the frozen
192-token ceiling would remove any content.

In [7]:
# Import the tokenizer class from the pinned Transformers release.
from transformers import AutoTokenizer

# Load tokenizer files from the immutable model revision.
tokenizer = AutoTokenizer.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_MODEL_REVISION,
    src_lang=SOURCE_LANG,
    tgt_lang=TARGET_LANG,
    cache_dir=os.environ["HF_HOME"],
)

def batched_token_lengths(texts, text_target=False, batch_size=256):
    '''Return untruncated encoded lengths in bounded-memory batches.'''
    lengths = []
    texts = [str(value) for value in texts]
    for start in range(0, len(texts), batch_size):
        batch = texts[start : start + batch_size]
        if text_target:
            encoded = tokenizer(text_target=batch, add_special_tokens=True, truncation=False)
        else:
            encoded = tokenizer(batch, add_special_tokens=True, truncation=False)
        lengths.extend(len(ids) for ids in encoded["input_ids"])
    return np.asarray(lengths, dtype=np.int32)

# Measure every source and target used by the experiment.
token_audits = {
    "train_source": batched_token_lengths(train_df["source_twi"]),
    "train_target": batched_token_lengths(train_df["target_english"], text_target=True),
    "dev_gold_source": batched_token_lengths(dev_df["source_gold_twi"]),
    "dev_mms_source": batched_token_lengths(dev_df["source_mms_twi"]),
    "dev_target": batched_token_lengths(dev_df["reference_english"] if "reference_english" in dev_df else dev_df["target_english"], text_target=True),
}

# Summarise distributions while retaining the strict maximum check.
token_audit_summary = {}
for name, values in token_audits.items():
    limit = TARGET_MAX_TOKENS if "target" in name else SOURCE_MAX_TOKENS
    token_audit_summary[name] = {
        "count": int(len(values)),
        "min": int(values.min()),
        "median": float(np.median(values)),
        "p95": float(np.percentile(values, 95)),
        "p99": float(np.percentile(values, 99)),
        "max": int(values.max()),
        "limit": int(limit),
        "would_truncate": int((values > limit).sum()),
    }
    assert token_audit_summary[name]["would_truncate"] == 0, token_audit_summary[name]

# Save and show the evidence before any optimization occurs.
write_json(OUTPUT_DIR / "TOKENIZER_LENGTH_AUDIT.json", token_audit_summary)
display(pd.DataFrame(token_audit_summary).T)

tokenizer_config.json:   0%|          | 0.00/564 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/4.85M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.3M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

,count,min,median,p95,p99,max,limit,would_truncate
train_source,7240.0,8.0,30.0,55.0,68.00,93.0,192.0,0.0
train_target,7240.0,8.0,22.0,40.0,50.61,73.0,192.0,0.0
dev_gold_source,1558.0,9.0,30.0,54.0,65.29,115.0,192.0,0.0
dev_mms_source,1558.0,9.0,28.0,50.0,61.00,113.0,192.0,0.0
dev_target,1558.0,9.0,22.0,41.0,49.00,85.0,192.0,0.0


## 6. Frozen metrics and deterministic helper functions

Corpus chrF++ is the primary automatic measure. BLEU and normalized
token F1 are supporting measures. Sentence chrF++ is used only for
paired diagnostic queues and the pre-specified material-regression rule.

In [8]:
# Configure the frozen metric implementations.
CHRF_METRIC = CHRF(word_order=2)
BLEU_METRIC = BLEU(tokenize="13a", effective_order=True)

def normalize_whitespace(text):
    '''Apply only Unicode-preserving whitespace normalization.'''
    return re.sub(r"\s+", " ", str(text)).strip()

def token_f1(reference, hypothesis):
    '''Compute multiset whitespace-token F1 without semantic rewriting.'''
    reference_tokens = normalize_whitespace(reference).casefold().split()
    hypothesis_tokens = normalize_whitespace(hypothesis).casefold().split()
    if not reference_tokens and not hypothesis_tokens:
        return 1.0
    if not reference_tokens or not hypothesis_tokens:
        return 0.0
    overlap = sum((Counter(reference_tokens) & Counter(hypothesis_tokens)).values())
    precision = overlap / len(hypothesis_tokens)
    recall = overlap / len(reference_tokens)
    return 0.0 if precision + recall == 0 else 2 * precision * recall / (precision + recall)

def corpus_metric_bundle(references, hypotheses):
    '''Calculate the frozen corpus and macro record-level measures.'''
    references = [normalize_whitespace(text) for text in references]
    hypotheses = [normalize_whitespace(text) for text in hypotheses]
    return {
        "chrf_pp": float(CHRF_METRIC.corpus_score(hypotheses, [references]).score),
        "sacrebleu": float(BLEU_METRIC.corpus_score(hypotheses, [references]).score),
        "macro_token_f1": float(np.mean([token_f1(r, h) for r, h in zip(references, hypotheses)])),
    }

def sentence_chrf(reference, hypothesis):
    '''Calculate record-level chrF++ for paired diagnostics.'''
    return float(CHRF_METRIC.sentence_score(normalize_whitespace(hypothesis), [normalize_whitespace(reference)]).score)

def seed_everything(seed):
    '''Set all available deterministic seeds for one training arm.'''
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    transformers.set_seed(seed)

def translate_sources(model, sources, batch_size=32):
    '''Generate English translations with the identical frozen decoder.'''
    model.eval()
    translations = []
    forced_bos = tokenizer.convert_tokens_to_ids(TARGET_LANG)
    for start in range(0, len(sources), batch_size):
        batch = [normalize_whitespace(value) for value in sources[start : start + batch_size]]
        encoded = tokenizer(
            batch,
            return_tensors="pt",
            padding=True,
            truncation=False,
        ).to(model.device)
        with torch.inference_mode():
            generated = model.generate(
                **encoded,
                forced_bos_token_id=forced_bos,
                num_beams=6,
                early_stopping=True,
                length_penalty=1.0,
                max_new_tokens=TARGET_MAX_TOKENS,
            )
        translations.extend(tokenizer.batch_decode(generated, skip_special_tokens=True))
    return [normalize_whitespace(text) for text in translations]

print("Frozen metric and decoding helpers are ready.")

Frozen metric and decoding helpers are ready.


## 7. Generate the frozen V3-M0 baseline once

Both baseline and adapted systems use the same pinned model revision,
tokenizer, and decoder. Baseline predictions are saved before training.

In [9]:
# Import the model class only when baseline generation begins.
from transformers import AutoModelForSeq2SeqLM

# Load the immutable unadapted model in A100/H100-native bfloat16.
seed_everything(BOOTSTRAP_SEED)
baseline_model = AutoModelForSeq2SeqLM.from_pretrained(
    BASE_MODEL_ID,
    revision=BASE_MODEL_REVISION,
    torch_dtype=torch.bfloat16,
    cache_dir=os.environ["HF_HOME"],
).cuda()

# Force the same target-language token used by the deployed path.
baseline_model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)

# Translate the paired gold and MMS development sources exactly once.
baseline_started = time.time()
dev_long["hypothesis_m0"] = translate_sources(
    baseline_model,
    dev_long["source_twi"].tolist(),
    batch_size=32,
)
baseline_seconds = time.time() - baseline_started

# Save row-level baseline predictions and condition-level metrics.
baseline_columns = [
    "record_uid", "content_group_id", "effective_content_group_id",
    "theme_key", "speaker_code", "input_condition", "source_twi",
    "reference_english", "hypothesis_m0", "dataset_row_sha256",
]
dev_long[baseline_columns].to_csv(OUTPUT_DIR / "V3_M0_BASELINE_PREDICTIONS.csv", index=False)

baseline_metrics = []
for condition, frame in dev_long.groupby("input_condition", sort=True):
    metrics = corpus_metric_bundle(frame["reference_english"], frame["hypothesis_m0"])
    baseline_metrics.append({"system": "V3-M0", "seed": None, "condition": condition, **metrics})
pd.DataFrame(baseline_metrics).to_csv(OUTPUT_DIR / "V3_M0_BASELINE_METRICS.csv", index=False)
write_json(
    OUTPUT_DIR / "V3_M0_RUNTIME.json",
    {"seconds": baseline_seconds, "rows": len(dev_long), "rows_per_second": len(dev_long) / baseline_seconds},
)
display(pd.DataFrame(baseline_metrics))

# Release baseline GPU memory before constructing the first LoRA arm.
del baseline_model
gc.collect()
torch.cuda.empty_cache()

config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


pytorch_model.bin:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.46G [00:00<?, ?B/s]

,system,seed,condition,chrf_pp,sacrebleu,macro_token_f1
0,V3-M0,None,gold,26.612235,7.443004,0.283469
1,V3-M0,None,mms,18.302801,1.612917,0.171677


## 8. Prepare group-balanced LoRA training

The custom trainer uses the precomputed `1 / group_variant_count`
weights. One epoch draws exactly 7,240 records with replacement, giving
each semantic group equal expected contribution.

In [10]:
# Import the pinned training, dataset, and PEFT components.
from datasets import Dataset
from peft import LoraConfig, TaskType, get_peft_model
from torch.utils.data import WeightedRandomSampler
from transformers import (
    DataCollatorForSeq2Seq,
    EarlyStoppingCallback,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments,
)
from transformers.models.m2m_100.modeling_m2m_100 import shift_tokens_right

# Convert data frames to Hugging Face datasets without hidden indexes.
raw_train_dataset = Dataset.from_pandas(train_df, preserve_index=False)
raw_eval_dataset = Dataset.from_pandas(
    dev_df.rename(columns={"source_gold_twi": "source_twi"}),
    preserve_index=False,
)

def preprocess_batch(batch):
    '''Tokenize source and target without allowing truncation.'''
    model_inputs = tokenizer(
        batch["source_twi"],
        max_length=SOURCE_MAX_TOKENS,
        truncation=False,
    )
    labels = tokenizer(
        text_target=batch["target_english"],
        max_length=TARGET_MAX_TOKENS,
        truncation=False,
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

# Tokenize once; fixed tokenization is shared across all seeds.
tokenized_train = raw_train_dataset.map(
    preprocess_batch,
    batched=True,
    desc="Tokenizing frozen train data",
)
tokenized_eval = raw_eval_dataset.map(
    preprocess_batch,
    batched=True,
    desc="Tokenizing frozen gold development data",
)

# Store weights outside the dataset so Trainer column-pruning cannot
# remove them before sampler construction.
sampling_weights = train_df["sampling_weight"].astype(float).to_numpy()

class GroupBalancedSeq2SeqTrainer(Seq2SeqTrainer):
    '''Seq2SeqTrainer with a deterministic, group-balanced sampler.'''

    def __init__(self, *args, sampling_weights, sampler_seed, **kwargs):
        self._sampling_weights = torch.as_tensor(sampling_weights, dtype=torch.double)
        self._sampler_seed = int(sampler_seed)
        super().__init__(*args, **kwargs)

    def _get_train_sampler(self, train_dataset=None):
        dataset = self.train_dataset if train_dataset is None else train_dataset
        assert len(dataset) == len(self._sampling_weights)
        generator = torch.Generator()
        generator.manual_seed(self._sampler_seed)
        return WeightedRandomSampler(
            weights=self._sampling_weights,
            num_samples=len(self._sampling_weights),
            replacement=True,
            generator=generator,
        )

class NLLBSeq2SeqCollator:
    '''Pad a seq2seq batch and explicitly construct M2M100 decoder inputs.'''

    def __init__(self, tokenizer, pad_token_id, decoder_start_token_id):
        # The NLLB/M2M100 model class has no
        # prepare_decoder_input_ids_from_labels method, so the stock
        # collator cannot construct decoder inputs for label smoothing.
        self._padding_collator = DataCollatorForSeq2Seq(
            tokenizer=tokenizer,
            model=None,
        )
        self._pad_token_id = int(pad_token_id)
        self._decoder_start_token_id = int(decoder_start_token_id)

    def __call__(self, features):
        batch = self._padding_collator(features)
        batch["decoder_input_ids"] = shift_tokens_right(
            batch["labels"],
            pad_token_id=self._pad_token_id,
            decoder_start_token_id=self._decoder_start_token_id,
        )
        return batch

def compute_eval_metrics(eval_prediction):
    '''Calculate checkpoint-selection chrF++ plus supporting measures.'''
    predictions, labels = eval_prediction
    if isinstance(predictions, tuple):
        predictions = predictions[0]
    labels = np.where(labels == -100, tokenizer.pad_token_id, labels)
    decoded_predictions = tokenizer.batch_decode(predictions, skip_special_tokens=True)
    decoded_labels = tokenizer.batch_decode(labels, skip_special_tokens=True)
    metrics = corpus_metric_bundle(decoded_labels, decoded_predictions)
    return {
        "chrf_pp": metrics["chrf_pp"],
        "sacrebleu": metrics["sacrebleu"],
        "macro_token_f1": metrics["macro_token_f1"],
    }

# Freeze the adapter architecture shared by all three seeds.
lora_config = LoraConfig(
    task_type=TaskType.SEQ_2_SEQ_LM,
    target_modules=["q_proj", "v_proj"],
    r=16,
    lora_alpha=32,
    lora_dropout=0.10,
    bias="none",
)

print("Group-balanced datasets and LoRA configuration are ready.")

Tokenizing frozen train data:   0%|          | 0/7240 [00:00<?, ? examples/s]

Tokenizing frozen gold development data:   0%|          | 0/1558 [00:00<?, ? examples/s]

Group-balanced datasets and LoRA configuration are ready.


## 9. Train seeds 17, 29, and 47 and generate full paired development outputs

This is the expensive cell. Each seed starts from the same immutable
base revision, saves epoch checkpoints and training history to Drive,
restores its best gold-development checkpoint, then translates both
the gold and raw MMS development conditions.

In [11]:
# Retain one combined frame so paired M0/M1 comparisons never rely on joins.
all_predictions = dev_long.copy()
seed_runtime_records = []

for seed in SEEDS:
    print(f"\n===== Starting V3-M1 seed {seed} =====")
    seed_everything(seed)
    seed_checkpoint_dir = CHECKPOINT_DIR / f"seed_{seed}"
    seed_adapter_dir = ADAPTER_DIR / f"seed_{seed}"
    seed_log_dir = LOG_DIR / f"seed_{seed}"
    seed_checkpoint_dir.mkdir(parents=True, exist_ok=True)
    seed_adapter_dir.mkdir(parents=True, exist_ok=True)
    seed_log_dir.mkdir(parents=True, exist_ok=True)

    # Reload the exact base weights for an independent seed replicate.
    base_model = AutoModelForSeq2SeqLM.from_pretrained(
        BASE_MODEL_ID,
        revision=BASE_MODEL_REVISION,
        torch_dtype=torch.bfloat16,
        cache_dir=os.environ["HF_HOME"],
    )
    base_model.generation_config.forced_bos_token_id = tokenizer.convert_tokens_to_ids(TARGET_LANG)
    model = get_peft_model(base_model, lora_config)
    model.print_trainable_parameters()

    # Record exact trainable/total parameter counts for the dissertation.
    trainable_parameters = sum(parameter.numel() for parameter in model.parameters() if parameter.requires_grad)
    total_parameters = sum(parameter.numel() for parameter in model.parameters())

    # Use only the hyperparameters frozen in the protocol.
    training_args = Seq2SeqTrainingArguments(
        output_dir=str(seed_checkpoint_dir),
        eval_strategy="epoch",
        save_strategy="epoch",
        learning_rate=2e-4,
        adam_beta1=0.9,
        adam_beta2=0.999,
        adam_epsilon=1e-8,
        weight_decay=0.01,
        lr_scheduler_type="linear",
        warmup_ratio=0.10,
        max_grad_norm=1.0,
        label_smoothing_factor=0.10,
        per_device_train_batch_size=16,
        per_device_eval_batch_size=32,
        gradient_accumulation_steps=2,
        num_train_epochs=8,
        bf16=True,
        tf32=True,
        predict_with_generate=True,
        generation_max_length=TARGET_MAX_TOKENS,
        generation_num_beams=6,
        load_best_model_at_end=True,
        metric_for_best_model="chrf_pp",
        greater_is_better=True,
        save_total_limit=2,
        logging_dir=str(seed_log_dir),
        logging_strategy="steps",
        logging_steps=25,
        report_to=["tensorboard"],
        seed=seed,
        data_seed=seed,
        optim="adamw_torch",
        dataloader_num_workers=2,
        remove_unused_columns=True,
    )

    # Explicitly shift target labels because NLLB's M2M100 implementation lacks
    # the collator hook required to do this automatically for label smoothing.
    data_collator = NLLBSeq2SeqCollator(
        tokenizer=tokenizer,
        pad_token_id=base_model.config.pad_token_id,
        decoder_start_token_id=base_model.config.decoder_start_token_id,
    )

    # Construct a trainer whose only nonstandard behavior is the frozen sampler.
    trainer = GroupBalancedSeq2SeqTrainer(
        model=model,
        args=training_args,
        train_dataset=tokenized_train,
        eval_dataset=tokenized_eval,
        processing_class=tokenizer,
        data_collator=data_collator,
        compute_metrics=compute_eval_metrics,
        callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
        sampling_weights=sampling_weights,
        sampler_seed=seed,
    )

    # Abort before any optimizer step if the collator/wrapper interface is invalid.
    smoke_batch = next(iter(trainer.get_train_dataloader()))
    assert "decoder_input_ids" in smoke_batch, sorted(smoke_batch.keys())
    assert "decoder_inputs_embeds" not in smoke_batch, sorted(smoke_batch.keys())
    smoke_device = next(model.parameters()).device
    smoke_inputs = {
        key: value.to(smoke_device)
        for key, value in smoke_batch.items()
        if key != "labels"
    }
    model.eval()
    with torch.no_grad():
        smoke_outputs = model(**smoke_inputs)
    assert torch.isfinite(smoke_outputs.logits).all(), "Non-finite pre-training logits."
    del smoke_batch, smoke_inputs, smoke_outputs
    model.train()
    seed_everything(seed)

    # Train, select the best gold-development checkpoint, and time the arm.
    train_started = time.time()
    train_result = trainer.train()
    training_seconds = time.time() - train_started
    assert np.isfinite(train_result.training_loss), "Non-finite training loss detected."

    # Save the selected LoRA adapter, tokenizer, logs, and trainer state.
    trainer.save_model(str(seed_adapter_dir))
    tokenizer.save_pretrained(str(seed_adapter_dir))
    trainer.save_state()
    write_json(seed_log_dir / "LOG_HISTORY.json", trainer.state.log_history)

    # Translate both development conditions with the restored best model.
    generation_started = time.time()
    seed_hypotheses = translate_sources(
        trainer.model,
        all_predictions["source_twi"].tolist(),
        batch_size=32,
    )
    generation_seconds = time.time() - generation_started
    all_predictions[f"hypothesis_s{seed}"] = seed_hypotheses

    # Record resource and checkpoint information without interpreting outcomes.
    seed_runtime = {
        "seed": seed,
        "training_seconds": training_seconds,
        "generation_seconds": generation_seconds,
        "best_checkpoint": trainer.state.best_model_checkpoint,
        "best_metric": trainer.state.best_metric,
        "training_loss": train_result.training_loss,
        "trainable_parameters": trainable_parameters,
        "total_parameters": total_parameters,
        "trainable_percent": 100.0 * trainable_parameters / total_parameters,
        "peak_gpu_memory_bytes": int(torch.cuda.max_memory_allocated()),
        "adapter_directory": str(seed_adapter_dir),
    }
    seed_runtime_records.append(seed_runtime)
    write_json(seed_log_dir / "SEED_RUNTIME.json", seed_runtime)

    # Save intermediate combined predictions so a later disconnect loses no seed.
    all_predictions.to_csv(OUTPUT_DIR / "V3_M0_M1_ALL_PREDICTIONS_PARTIAL.csv", index=False)

    # Release this seed's optimizer and model before the next independent arm.
    del trainer, model, base_model, data_collator
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

# Persist the completed three-seed row-level prediction table.
all_predictions.to_csv(OUTPUT_DIR / "V3_M0_M1_ALL_PREDICTIONS.csv", index=False)
pd.DataFrame(seed_runtime_records).to_csv(OUTPUT_DIR / "V3_M1_SEED_RUNTIME.csv", index=False)
display(pd.DataFrame(seed_runtime_records))


===== Starting V3-M1 seed 17 =====
trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Memory Efficient attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:900.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
/usr/local/lib/python3.12/dist-packages/torch/autograd/graph.py:869: UserWarning: Flash Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at /pytorch/aten/src/ATen/native/transformers/cuda/attention_backward.cu:124.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


Epoch,Training Loss,Validation Loss,Chrf Pp,Sacrebleu,Macro Token F1
1,3.344400,3.240288,35.390954,16.141888,0.397973
2,3.160000,3.126228,37.093044,17.770149,0.418655
3,3.098100,3.085954,38.118309,18.760635,0.429475
4,3.079200,3.055145,38.806033,19.301961,0.434355
5,3.008700,3.045418,39.451113,19.981599,0.442011
6,2.998300,3.035073,39.314436,19.812295,0.440776
7,2.977900,3.028847,39.861632,20.205184,0.445281
8,3.011500,3.027549,39.910976,20.283938,0.447558


config.json:   0%|          | 0.00/846 [00:00<?, ?B/s]


===== Starting V3-M1 seed 29 =====
trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


Epoch,Training Loss,Validation Loss,Chrf Pp,Sacrebleu,Macro Token F1
1,3.354300,3.246703,35.163253,15.667572,0.395455
2,3.193100,3.126075,37.458394,17.977254,0.418235
3,3.105600,3.085436,38.392849,18.761969,0.431806
4,3.062100,3.058787,39.104600,19.324837,0.439148
5,3.012600,3.042096,39.320300,19.586716,0.439981
6,3.023700,3.031433,39.835797,20.057485,0.444469
7,2.981700,3.026887,39.726492,19.929548,0.445371
8,2.965300,3.022175,39.803383,19.939367,0.445439



===== Starting V3-M1 seed 47 =====
trainable params: 2,359,296 || all params: 617,433,088 || trainable%: 0.3821


Epoch,Training Loss,Validation Loss,Chrf Pp,Sacrebleu,Macro Token F1
1,3.332300,3.240347,34.874237,15.430245,0.391364
2,3.171500,3.124452,36.911313,17.560601,0.413030
3,3.098000,3.082619,38.112996,18.714492,0.427697
4,3.057000,3.058410,38.973721,19.460312,0.437506
5,2.984700,3.041524,39.008501,19.364856,0.438486
6,2.952200,3.030095,39.466591,19.984534,0.441969
7,2.997600,3.025593,39.755742,20.059717,0.446715
8,2.987800,3.022642,39.748638,20.019492,0.446714


,seed,training_seconds,generation_seconds,best_checkpoint,best_metric,training_loss,trainable_parameters,total_parameters,trainable_percent,peak_gpu_memory_bytes,adapter_directory
0,17,1753.529704,206.830115,/content/drive/MyDrive/Akan_ASR_PhD_Experiment...,39.910976,3.127619,2359296,617433088,0.382114,10026625024,/content/drive/MyDrive/Akan_ASR_PhD_Experiment...
1,29,1781.996557,227.797488,/content/drive/MyDrive/Akan_ASR_PhD_Experiment...,39.835797,3.132336,2359296,617433088,0.382114,10026625024,/content/drive/MyDrive/Akan_ASR_PhD_Experiment...
2,47,1751.859409,203.814587,/content/drive/MyDrive/Akan_ASR_PhD_Experiment...,39.755742,3.132608,2359296,617433088,0.382114,10026628096,/content/drive/MyDrive/Akan_ASR_PhD_Experiment...


## 10. Calculate paired point estimates and material regressions

The table below is descriptive. Promotion cannot be decided until the
clustered intervals and human semantic-safety review are complete.

In [12]:
# Calculate system-level point estimates for every input condition.
point_estimates = []
paired_diagnostics = []

for condition, condition_frame in all_predictions.groupby("input_condition", sort=True):
    baseline_bundle = corpus_metric_bundle(
        condition_frame["reference_english"],
        condition_frame["hypothesis_m0"],
    )
    point_estimates.append({"system": "V3-M0", "seed": None, "condition": condition, **baseline_bundle})

    for seed in SEEDS:
        hypothesis_column = f"hypothesis_s{seed}"
        adapted_bundle = corpus_metric_bundle(
            condition_frame["reference_english"],
            condition_frame[hypothesis_column],
        )
        point_estimates.append({
            "system": "V3-M1",
            "seed": seed,
            "condition": condition,
            **adapted_bundle,
            "delta_chrf_pp": adapted_bundle["chrf_pp"] - baseline_bundle["chrf_pp"],
            "delta_sacrebleu": adapted_bundle["sacrebleu"] - baseline_bundle["sacrebleu"],
            "delta_macro_token_f1": adapted_bundle["macro_token_f1"] - baseline_bundle["macro_token_f1"],
        })

        # Retain per-record differences for the frozen regression threshold.
        for index, row in condition_frame.iterrows():
            baseline_sentence = sentence_chrf(row["reference_english"], row["hypothesis_m0"])
            adapted_sentence = sentence_chrf(row["reference_english"], row[hypothesis_column])
            paired_diagnostics.append({
                "row_index": int(index),
                "record_uid": row["record_uid"],
                "content_group_id": row["content_group_id"],
                "effective_content_group_id": row["effective_content_group_id"],
                "theme_key": row["theme_key"],
                "speaker_code": row["speaker_code"],
                "condition": condition,
                "seed": seed,
                "m0_sentence_chrf_pp": baseline_sentence,
                "m1_sentence_chrf_pp": adapted_sentence,
                "delta_sentence_chrf_pp": adapted_sentence - baseline_sentence,
                "material_regression": (adapted_sentence - baseline_sentence) <= -5.0,
            })

point_estimates_df = pd.DataFrame(point_estimates)
paired_diagnostics_df = pd.DataFrame(paired_diagnostics)
point_estimates_df.to_csv(OUTPUT_DIR / "POINT_ESTIMATES.csv", index=False)
paired_diagnostics_df.to_csv(OUTPUT_DIR / "PAIRED_RECORD_DIAGNOSTICS.csv", index=False)

# Summarise the pre-specified 5-point regression threshold by seed/condition.
regression_summary = (
    paired_diagnostics_df.groupby(["condition", "seed"], as_index=False)
    .agg(records=("record_uid", "size"), material_regressions=("material_regression", "sum"))
)
regression_summary["material_regression_rate"] = (
    regression_summary["material_regressions"] / regression_summary["records"]
)
regression_summary.to_csv(OUTPUT_DIR / "MATERIAL_REGRESSION_SUMMARY.csv", index=False)
display(point_estimates_df)
display(regression_summary)

,system,seed,condition,chrf_pp,sacrebleu,macro_token_f1,delta_chrf_pp,delta_sacrebleu,delta_macro_token_f1
0,V3-M0,NaN,gold,26.612235,7.443004,0.283469,NaN,NaN,NaN
1,V3-M1,17.0,gold,39.748316,20.175346,0.446877,13.136081,12.732342,0.163408
2,V3-M1,29.0,gold,39.680287,19.904941,0.444552,13.068052,12.461938,0.161083
3,V3-M1,47.0,gold,39.586838,19.903011,0.445897,12.974603,12.460007,0.162428
4,V3-M0,NaN,mms,18.302801,1.612917,0.171677,NaN,NaN,NaN
5,V3-M1,17.0,mms,25.484108,7.202936,0.267688,7.181307,5.590018,0.096012
6,V3-M1,29.0,mms,25.430070,7.178350,0.264277,7.127269,5.565433,0.092600
7,V3-M1,47.0,mms,25.364747,7.239107,0.266026,7.061946,5.626189,0.094349


,condition,seed,records,material_regressions,material_regression_rate
0,gold,17,1558,82,0.052632
1,gold,29,1558,72,0.046213
2,gold,47,1558,79,0.050706
3,mms,17,1558,101,0.064827
4,mms,29,1558,94,0.060334
5,mms,47,1558,108,0.069320


## 11. Run the 20,000-replicate paired group-cluster bootstrap

Resampling uses the 458 original `content_group_id` clusters. The code
pre-aggregates chrF sufficient statistics by cluster, verifies that
their sum reproduces the public corpus score, then resamples clusters.
This is much faster than repeatedly tokenizing all 3,116 long-form rows.

In [13]:
def chrf_segment_statistics(hypotheses, references):
    '''Extract sacreBLEU chrF sufficient statistics per record.'''
    statistics = CHRF_METRIC._extract_corpus_statistics(
        [normalize_whitespace(text) for text in hypotheses],
        [[normalize_whitespace(text) for text in references]],
    )
    return np.asarray(statistics, dtype=np.float64)

def score_from_chrf_statistics(statistics):
    '''Convert one summed chrF statistic vector to its 0–100 score.'''
    return float(CHRF_METRIC._compute_score_from_stats(statistics).score)

def clustered_chrf_bootstrap(condition_frame, condition, chunk_size=500):
    '''Return seed-specific and pooled paired chrF differences.'''
    groups = sorted(condition_frame["content_group_id"].astype(str).unique())
    group_to_position = {group: position for position, group in enumerate(groups)}
    group_positions = condition_frame["content_group_id"].astype(str).map(group_to_position).to_numpy()
    references = condition_frame["reference_english"].tolist()

    # Extract baseline and adapted sufficient statistics once.
    system_columns = ["hypothesis_m0"] + [f"hypothesis_s{seed}" for seed in SEEDS]
    row_statistics = {
        column: chrf_segment_statistics(condition_frame[column].tolist(), references)
        for column in system_columns
    }

    # Verify the private sufficient-statistic path exactly matches the
    # public corpus score before using it for resampling.
    for column in system_columns:
        public_score = CHRF_METRIC.corpus_score(condition_frame[column].tolist(), [references]).score
        statistics_score = score_from_chrf_statistics(row_statistics[column].sum(axis=0))
        assert abs(public_score - statistics_score) < 1e-9, (column, public_score, statistics_score)

    # Aggregate every statistic dimension by original content group.
    grouped_statistics = {}
    for column, statistics in row_statistics.items():
        grouped = np.zeros((len(groups), statistics.shape[1]), dtype=np.float64)
        np.add.at(grouped, group_positions, statistics)
        grouped_statistics[column] = grouped

    # Draw all cluster multiplicities from the single frozen RNG stream.
    condition_offset = 0 if condition == "gold" else 1
    rng = np.random.default_rng(BOOTSTRAP_SEED + condition_offset)
    bootstrap_rows = []

    for start in range(0, BOOTSTRAP_REPLICATES, chunk_size):
        current_size = min(chunk_size, BOOTSTRAP_REPLICATES - start)
        multiplicities = rng.multinomial(
            n=len(groups),
            pvals=np.full(len(groups), 1.0 / len(groups)),
            size=current_size,
        )
        summed = {
            column: multiplicities @ grouped_statistics[column]
            for column in system_columns
        }

        for local_index in range(current_size):
            baseline_score = score_from_chrf_statistics(summed["hypothesis_m0"][local_index])
            seed_differences = []
            record = {
                "condition": condition,
                "replicate": start + local_index,
                "m0_chrf_pp": baseline_score,
            }
            for seed in SEEDS:
                adapted_score = score_from_chrf_statistics(
                    summed[f"hypothesis_s{seed}"][local_index]
                )
                difference = adapted_score - baseline_score
                seed_differences.append(difference)
                record[f"delta_s{seed}"] = difference
            record["delta_three_seed_mean"] = float(np.mean(seed_differences))
            bootstrap_rows.append(record)

    return pd.DataFrame(bootstrap_rows)

# Run and save the exact frozen bootstrap for both input conditions.
bootstrap_frames = []
for condition, frame in all_predictions.groupby("input_condition", sort=True):
    bootstrap_frames.append(clustered_chrf_bootstrap(frame.reset_index(drop=True), condition))
bootstrap_df = pd.concat(bootstrap_frames, ignore_index=True)
bootstrap_df.to_csv(OUTPUT_DIR / "CLUSTER_BOOTSTRAP_20000.csv", index=False)

# Calculate percentile intervals without parametric assumptions.
bootstrap_summary = []
for condition, frame in bootstrap_df.groupby("condition", sort=True):
    for column in [f"delta_s{seed}" for seed in SEEDS] + ["delta_three_seed_mean"]:
        values = frame[column].to_numpy()
        bootstrap_summary.append({
            "condition": condition,
            "estimate": column,
            "bootstrap_mean": float(values.mean()),
            "ci95_lower": float(np.percentile(values, 2.5)),
            "ci95_upper": float(np.percentile(values, 97.5)),
            "replicates": len(values),
            "cluster_count": all_predictions.loc[
                all_predictions["input_condition"] == condition,
                "content_group_id",
            ].nunique(),
        })
bootstrap_summary_df = pd.DataFrame(bootstrap_summary)
bootstrap_summary_df.to_csv(OUTPUT_DIR / "CLUSTER_BOOTSTRAP_SUMMARY.csv", index=False)
display(bootstrap_summary_df)

,condition,estimate,bootstrap_mean,ci95_lower,ci95_upper,replicates,cluster_count
0,gold,delta_s17,13.136902,12.143328,14.148998,20000,458
1,gold,delta_s29,13.069103,12.066897,14.095261,20000,458
2,gold,delta_s47,12.976831,11.993760,13.973175,20000,458
3,gold,delta_three_seed_mean,13.060945,12.093334,14.047119,20000,458
4,mms,delta_s17,7.180130,6.527622,7.844258,20000,458
5,mms,delta_s29,7.125295,6.482120,7.791658,20000,458
6,mms,delta_s47,7.061722,6.375550,7.777734,20000,458
7,mms,delta_three_seed_mean,7.122382,6.499805,7.773282,20000,458


## 12. Build the blinded semantic-safety review queue

The frozen lexicon is a high-recall trigger, not an ontology. Every
material regression and clinical-concept disagreement is included,
plus a deterministic 10% stratified sample of remaining changed rows.
Candidate identities are randomized and stored in a separate reveal key.

In [14]:
# Normalize the frozen category dictionaries for transparent matching.
english_categories = clinical_lexicon["english_categories"]
akan_categories = clinical_lexicon["akan_source_categories"]
number_pattern = re.compile(clinical_lexicon["number_pattern"], flags=re.IGNORECASE)

def contains_term(text, term):
    '''Use boundary-padded substring matching without rewriting words.'''
    padded_text = f" {normalize_whitespace(text).casefold()} "
    padded_term = f" {normalize_whitespace(term).casefold()} "
    return padded_term in padded_text

def matched_categories(text, category_dictionary):
    '''Return every frozen semantic category represented in a string.'''
    return {
        category
        for category, terms in category_dictionary.items()
        if any(contains_term(text, term) for term in terms)
    }

def matched_terms(text, category_dictionary):
    '''Return explicit matched terms to make each trigger auditable.'''
    return sorted({
        term
        for terms in category_dictionary.values()
        for term in terms
        if contains_term(text, term)
    })

# Index the material-regression flag by original long-frame row and seed.
regression_lookup = {
    (int(row.row_index), int(row.seed)): bool(row.material_regression)
    for row in paired_diagnostics_df.itertuples(index=False)
}

review_candidates = []
changed_nontriggered = []

for row_index, row in all_predictions.iterrows():
    source_categories = matched_categories(row["source_twi"], akan_categories)
    reference_categories = matched_categories(row["reference_english"], english_categories)
    m0_categories = matched_categories(row["hypothesis_m0"], english_categories)

    for seed in SEEDS:
        m1_text = row[f"hypothesis_s{seed}"]
        if normalize_whitespace(m1_text) == normalize_whitespace(row["hypothesis_m0"]):
            continue

        m1_categories = matched_categories(m1_text, english_categories)
        triggers = []
        if regression_lookup[(int(row_index), seed)]:
            triggers.append("MATERIAL_REGRESSION_LE_5_CHRF")
        if m0_categories != m1_categories:
            triggers.append("M0_M1_CLINICAL_CATEGORY_DISAGREEMENT")
        if source_categories and not (source_categories & m1_categories):
            triggers.append("SOURCE_CLINICAL_CATEGORY_MISSING_IN_M1")
        if m1_categories - (reference_categories | m0_categories):
            triggers.append("M1_CLINICAL_CATEGORY_NOVELTY")
        if bool(number_pattern.search(str(row["hypothesis_m0"]))) != bool(number_pattern.search(str(m1_text))):
            triggers.append("M0_M1_NUMBER_DISAGREEMENT")

        candidate = {
            "row_index": int(row_index),
            "record_uid": row["record_uid"],
            "content_group_id": row["content_group_id"],
            "effective_content_group_id": row["effective_content_group_id"],
            "theme_key": row["theme_key"],
            "speaker_code": row["speaker_code"],
            "input_condition": row["input_condition"],
            "seed": seed,
            "source_twi": row["source_twi"],
            "reference_english": row["reference_english"],
            "m0_text": row["hypothesis_m0"],
            "m1_text": m1_text,
            "trigger_reasons": ";".join(sorted(set(triggers))),
            "source_categories": ";".join(sorted(source_categories)),
            "reference_categories": ";".join(sorted(reference_categories)),
            "m0_categories": ";".join(sorted(m0_categories)),
            "m1_categories": ";".join(sorted(m1_categories)),
            "m0_matched_terms": ";".join(matched_terms(row["hypothesis_m0"], english_categories)),
            "m1_matched_terms": ";".join(matched_terms(m1_text, english_categories)),
        }
        if triggers:
            review_candidates.append(candidate)
        else:
            changed_nontriggered.append(candidate)

# Add a fixed 10% sample of otherwise non-triggered changed rows,
# stratified by theme, input condition, and seed.
nontriggered_df = pd.DataFrame(changed_nontriggered)
sampled_frames = []
if not nontriggered_df.empty:
    for _, stratum in nontriggered_df.groupby(["theme_key", "input_condition", "seed"], dropna=False):
        sample_size = max(1, int(np.ceil(0.10 * len(stratum))))
        sampled_frames.append(stratum.sample(n=sample_size, random_state=BOOTSTRAP_SEED))
sampled_df = pd.concat(sampled_frames, ignore_index=True) if sampled_frames else pd.DataFrame()
if not sampled_df.empty:
    sampled_df["trigger_reasons"] = "STRATIFIED_10_PERCENT_CHANGED_SAMPLE"

# Combine all pre-specified triggers and remove accidental duplicates.
triggered_df = pd.DataFrame(review_candidates)
review_df = pd.concat([triggered_df, sampled_df], ignore_index=True)
review_df = review_df.drop_duplicates(subset=["row_index", "seed"], keep="first")

# Deterministically randomize M0/M1 into candidate A/B for blinded review.
blinded_rows = []
reveal_rows = []
for row in review_df.itertuples(index=False):
    key = f"{row.record_uid}|{row.input_condition}|{row.seed}|{BOOTSTRAP_SEED}"
    m0_is_a = int(hashlib.sha256(key.encode("utf-8")).hexdigest(), 16) % 2 == 0
    candidate_a = row.m0_text if m0_is_a else row.m1_text
    candidate_b = row.m1_text if m0_is_a else row.m0_text
    review_id = hashlib.sha256(key.encode("utf-8")).hexdigest()[:16]
    blinded_rows.append({
        "review_id": review_id,
        "record_uid": row.record_uid,
        "content_group_id": row.content_group_id,
        "effective_content_group_id": row.effective_content_group_id,
        "theme_key": row.theme_key,
        "speaker_code": row.speaker_code,
        "input_condition": row.input_condition,
        "source_twi": row.source_twi,
        "reference_english": row.reference_english,
        "candidate_a": candidate_a,
        "candidate_b": candidate_b,
        "trigger_reasons": row.trigger_reasons,
        "intent_preservation_a": "",
        "intent_preservation_b": "",
        "question_statement_preserved_a": "",
        "question_statement_preserved_b": "",
        "participant_preserved_a": "",
        "participant_preserved_b": "",
        "requested_action_preserved_a": "",
        "requested_action_preserved_b": "",
        "negation_preserved_a": "",
        "negation_preserved_b": "",
        "number_preserved_a": "",
        "number_preserved_b": "",
        "critical_concept_preserved_a": "",
        "critical_concept_preserved_b": "",
        "severity_a": "",
        "severity_b": "",
        "preferred_candidate": "",
        "adjudication_note": "",
    })
    reveal_rows.append({
        "review_id": review_id,
        "seed": int(row.seed),
        "candidate_a_system": "V3-M0" if m0_is_a else f"V3-M1-S{row.seed}",
        "candidate_b_system": f"V3-M1-S{row.seed}" if m0_is_a else "V3-M0",
    })

blinded_review_df = pd.DataFrame(blinded_rows)
reveal_key_df = pd.DataFrame(reveal_rows)
blinded_review_df.to_csv(REVIEW_DIR / "SEMANTIC_SAFETY_REVIEW_QUEUE_BLINDED.csv", index=False)
reveal_key_df.to_csv(REVIEW_DIR / "SEMANTIC_SAFETY_REVEAL_KEY_DO_NOT_OPEN_BEFORE_REVIEW.csv", index=False)

review_queue_summary = {
    "changed_triggered_rows": int(len(triggered_df)),
    "sampled_nontriggered_rows": int(len(sampled_df)),
    "final_unique_review_rows": int(len(blinded_review_df)),
    "human_review_complete": False,
    "promotion_decision": "NOT_PERMITTED_BEFORE_REVIEW",
}
write_json(REVIEW_DIR / "REVIEW_QUEUE_SUMMARY.json", review_queue_summary)
display(pd.DataFrame([review_queue_summary]))
display(blinded_review_df.head(10))

,changed_triggered_rows,sampled_nontriggered_rows,final_unique_review_rows,human_review_complete,promotion_decision
0,6841,307,7148,False,NOT_PERMITTED_BEFORE_REVIEW


,review_id,record_uid,content_group_id,effective_content_group_id,theme_key,speaker_code,input_condition,source_twi,reference_english,candidate_a,...,negation_preserved_a,negation_preserved_b,number_preserved_a,number_preserved_b,critical_concept_preserved_a,critical_concept_preserved_b,severity_a,severity_b,preferred_candidate,adjudication_note
0,61c7ed6cc4c70888,BT_0029,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Home akwan bi wɔ hɔ a mɛtumi afa so ahome ama ...,Are there any breathing techniques to try duri...,Is there any way I can breathe to help me duri...,...,,,,,,,,,,
1,6efec0b94252dd8b,BT_0029,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Home akwan bi wɔ hɔ a mɛtumi afa so ahome ama ...,Are there any breathing techniques to try duri...,Is there any way I can breathe to help me duri...,...,,,,,,,,,,
2,89e5cb63ad7af7cb,BT_0029,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Home akwan bi wɔ hɔ a mɛtumi afa so ahome ama ...,Are there any breathing techniques to try duri...,Is there any way I can breathe to help me duri...,...,,,,,,,,,,
3,749975c83c7c0906,BT_0030,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Yɛwɔ kwan sononko bi wɔ hɔ a ɔpemfoɔ bɛtumi af...,Are there any breathing techniques to try duri...,Do we have a different way for a midwife to br...,...,,,,,,,,,,
4,352606053b4b608f,BT_0030,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Yɛwɔ kwan sononko bi wɔ hɔ a ɔpemfoɔ bɛtumi af...,Are there any breathing techniques to try duri...,Do we have a different way for a midwife to br...,...,,,,,,,,,,
5,c52668b0527bcbf8,BT_0030,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Yɛwɔ kwan sononko bi wɔ hɔ a ɔpemfoɔ bɛtumi af...,Are there any breathing techniques to try duri...,Is there a special way to breathe during pregn...,...,,,,,,,,,,
6,027746e0d5c5021d,BT_0031,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Kwan bɛn so na ɔbaa a ɔnyem nam home so bɛtumi...,Are there any breathing techniques to try duri...,Is it safe to breathe during pregnancy?,...,,,,,,,,,,
7,b9d52327a3bb6097,BT_0031,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Kwan bɛn so na ɔbaa a ɔnyem nam home so bɛtumi...,Are there any breathing techniques to try duri...,How can breastfeeding help during pregnancy?,...,,,,,,,,,,
8,2971fd8f038ddcd6,BT_0031,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Kwan bɛn so na ɔbaa a ɔnyem nam home so bɛtumi...,Are there any breathing techniques to try duri...,How can a breastfeeding woman help herself dur...,...,,,,,,,,,,
9,0d5b9c583ea8825a,BT_0032,CG00008,CG00008,prenatal_physical_exercise,BT,gold,Home mu dwumadie bɛn na ɛbɛboa ɔbaa a ɔnyem?,Are there any breathing techniques to try duri...,What breathing exercises can help during pregn...,...,,,,,,,,,,


## 13. Evaluate automatic gates and close the development run

Passing these automatic checks does not promote the model. The blinded
semantic-safety review remains mandatory, followed by a paired SBLLM
development evaluation. The sealed test remains unopened.

In [15]:
# Pull the frozen three-seed confidence intervals by condition.
pooled_rows = bootstrap_summary_df[
    bootstrap_summary_df["estimate"] == "delta_three_seed_mean"
].set_index("condition")

# Pull seed-specific point differences for the primary MMS condition.
mms_seed_deltas = (
    point_estimates_df[
        (point_estimates_df["system"] == "V3-M1")
        & (point_estimates_df["condition"] == "mms")
    ]
    .set_index("seed")["delta_chrf_pp"]
    .to_dict()
)

# Calculate the strict automatic gate components exactly as frozen.
automatic_gates = {
    "primary_mms_ci_lower_ge_0": bool(pooled_rows.loc["mms", "ci95_lower"] >= 0.0),
    "gold_noninferiority_ci_lower_ge_minus_0_5": bool(pooled_rows.loc["gold", "ci95_lower"] >= -0.5),
    "all_seed_mms_point_deltas_positive": bool(all(value > 0.0 for value in mms_seed_deltas.values())),
    "mms_seed_delta_range_le_2": bool((max(mms_seed_deltas.values()) - min(mms_seed_deltas.values())) <= 2.0),
    "material_regression_rate_le_5_percent_every_arm": bool(
        (regression_summary["material_regression_rate"] <= 0.05).all()
    ),
}
automatic_gates["all_automatic_gates_pass"] = bool(all(automatic_gates.values()))

# A complete computation is still explicitly held for human review.
final_status = {
    "protocol_id": PROTOCOL_ID,
    "protocol_sha256": PROTOCOL_SHA256,
    "run_id": RUN_ID,
    "status": "DEVELOPMENT_COMPUTATION_COMPLETE",
    "automatic_gates": automatic_gates,
    "human_semantic_safety_review": "PENDING",
    "downstream_sbllm_evaluation": "NOT_RUN",
    "sealed_test_opened": False,
    "test_rows_read": 0,
    "production_changed": False,
    "promotion_decision": "NOT_MADE",
    "completed_utc": datetime.now(timezone.utc).isoformat(),
}
write_json(RUN_DIR / "RUN_COMPLETE.json", final_status)

# Hash every completed artifact, excluding the checksum file itself.
checksum_rows = []
for path in sorted(RUN_DIR.rglob("*")):
    if path.is_file() and path.name not in {"SHA256SUMS.txt", "RUN_INCOMPLETE.json"}:
        checksum_rows.append(f"{sha256_file(path)}  {path.relative_to(RUN_DIR).as_posix()}")
(RUN_DIR / "SHA256SUMS.txt").write_text("\n".join(checksum_rows) + "\n", encoding="utf-8")

# Mark the notebook complete and remove the provisional incomplete record.
RUN_STATE["complete"] = True
incomplete_path = RUN_DIR / "RUN_INCOMPLETE.json"
if incomplete_path.exists():
    incomplete_path.unlink()

print(json.dumps(final_status, indent=2))
print(f"\nBlinded review queue: {REVIEW_DIR / 'SEMANTIC_SAFETY_REVIEW_QUEUE_BLINDED.csv'}")
print("STOP HERE: do not open the reveal key, sealed test, or deploy a model before adjudication.")

{
  "protocol_id": "nllb-v3-lora-dev-v1",
  "protocol_sha256": "242297bc42129587d84a9a3ca51a0b6105ac42181ffb706cc021202e3703a9d9",
  "run_id": "20260804T191924Z_nllb_v3_lora_dev_v1",
  "status": "DEVELOPMENT_COMPUTATION_COMPLETE",
  "automatic_gates": {
    "primary_mms_ci_lower_ge_0": true,
    "gold_noninferiority_ci_lower_ge_minus_0_5": true,
    "all_seed_mms_point_deltas_positive": true,
    "mms_seed_delta_range_le_2": true,
    "material_regression_rate_le_5_percent_every_arm": false,
    "all_automatic_gates_pass": false
  },
  "human_semantic_safety_review": "PENDING",
  "downstream_sbllm_evaluation": "NOT_RUN",
  "sealed_test_opened": false,
  "test_rows_read": 0,
  "production_changed": false,
  "promotion_decision": "NOT_MADE",
  "completed_utc": "2026-08-04T21:04:08.014865+00:00"
}

Blinded review queue: /content/drive/MyDrive/Akan_ASR_PhD_Experiments/03_Adaptation/nllb_v3_2026-08-04/runs/20260804T191924Z_nllb_v3_lora_dev_v1/human_review/SEMANTIC_SAFETY_REVIEW_QUEUE_BLINDE

## Required next action after this notebook

1. Review every row in `SEMANTIC_SAFETY_REVIEW_QUEUE_BLINDED.csv` without
   opening the reveal key.
2. Freeze and hash the completed adjudication.
3. Apply the semantic-safety gates and reveal the system identities.
4. If and only if all translation gates pass, run the paired downstream
   SBLLM development evaluation.
5. Keep the final test sealed until the signed downstream release record.

A positive chrF++ result alone is not sufficient for promotion.